# Utility-Scale Solar Suitability Analysis for Minnesota

A GIS-based multi-criteria decision analysis (MCDA) identifying the most promising parcels for utility-scale solar development statewide. Two hard constraints (land cover, minimum parcel size) exclude clearly unsuitable land outright; three weighted suitability factors (land cover quality, slope, south-facing aspect) and a road-density penalty are then combined into a single ranked score, following the standard weighted linear combination approach to GIS suitability modeling (see Malczewski, 2004, *GIS-based land-use suitability analysis: a critical overview*, for a review of this method).

**On reproducing this:** like the other ArcPy-based projects in this portfolio, this pipeline runs inside ArcGIS Pro and needs a Spatial Analyst license - it isn't something you can clone and run outside that environment, and the source rasters/parcel data aren't included here.

## Setup

In [ ]:
import arcpy
from arcpy import env
from arcpy.sa import *

# update this to your own project geodatabase
output_gdb = r"C:\path\to\your\project\SolarSuitability.gdb"
env.workspace = output_gdb
arcpy.env.overwriteOutput = True

## Part 1: Hard constraints

Two binary filters are applied first, ahead of any weighted scoring: land cover type and minimum parcel size. Land that fails either constraint is excluded from consideration entirely, rather than being scored down - these aren't factors to be traded off, they're outright disqualifiers.

### Land cover

Land cover comes from the NLCD raster, reclassified to a binary suitability layer: NLCD classes 21 (Developed, Open Space), 52 (Shrub/Scrub), 71 (Grassland/Herbaceous), 81 (Pasture/Hay), and 82 (Cultivated Crops) are treated as suitable (1); everything else is unsuitable (0).

**Reproducibility note:** this step was originally done by hand in ArcGIS Pro's UI (Build Raster Attribute Table, then a point-and-click reclassify), since the arcpy version wasn't cooperating at the time. The cell below is a code equivalent of that manual workflow, using the same class codes - it reproduces the *documented* logic, but hasn't been independently re-run to confirm it produces an identical output to the original manual pass. Worth verifying against the original `Reclass_NLCD1` raster before relying on it.

In [ ]:
nlcd_raster = r"C:\path\to\your\NLCD_MN.tif"

suitable_classes = [21, 52, 71, 81, 82]
remap_list = [[c, 1] for c in suitable_classes]

lc_binary = Reclassify(
    nlcd_raster,
    "Value",
    RemapValue(remap_list),
    missing_values="NODATA"
)

lc_binary.save(f"{output_gdb}\\Reclass_NLCD1")

### Parcel size

Parcels are limited to those at least 30 acres, using Minnesota's opt-in county parcel dataset.

**Coverage limitation, stated plainly:** only 59 of Minnesota's 87 counties currently participate in this open parcel program. Parcels in the other 28 counties simply don't exist in this dataset and can never appear in the results below - the top-10 ranking that comes out of this pipeline is a top 10 *among participating counties*, not the state as a whole. That's a real constraint on what this analysis can claim, not a minor caveat.

In [9]:
parcels_src = r"C:\path\to\your\plan_parcels_open.gdb\plan_parcels_open"

arcpy.management.MakeFeatureLayer(parcels_src, "parcels_lyr")
arcpy.management.SelectLayerByAttribute("parcels_lyr", "NEW_SELECTION", "acres_poly >= 30")

parcels = f"{output_gdb}\\parcels_30acres"
arcpy.management.CopyFeatures("parcels_lyr", parcels)

<Result ''>

## Part 2: Scoring land cover suitability per parcel

With the hard constraints applied, suitability scoring begins. For each parcel, the percentage of its area classified as suitable land cover is calculated via `TabulateArea`, then joined back onto the parcel layer as `pct_suitable`.

In [37]:
lc_binary_path = f"{output_gdb}\\Reclass_NLCD1"
tab_area = f"{output_gdb}\\lc_tabulate_area"

TabulateArea(
    in_zone_data=parcels,
    zone_field="OBJECTID",
    in_class_data=lc_binary_path,
    class_field="Value",
    out_table=tab_area,
    processing_cell_size=30)

<Result ''>

In [38]:
#add field for pct suitable
arcpy.management.AddField(
    in_table=tab_area,
    field_name="pct_suitable",
    field_type="DOUBLE")

<Result ''>

In [39]:
# percent of parcel area classified as suitable land cover
expression = "(!VALUE_1! / (!VALUE_0! + !VALUE_1!)) * 100"

arcpy.management.CalculateField(
    in_table=tab_area,
    field="pct_suitable",
    expression=expression,
    expression_type="PYTHON3")

<Result ''>

In [40]:
#join back to parcel 
arcpy.management.JoinField(
    in_data=parcels,
    in_field="OBJECTID",
    join_table=tab_area,
    join_field="OBJECTID",
    fields=["pct_suitable"]
)

<Result ''>

## Part 3: Terrain suitability

Two terrain factors are derived from the digital elevation model (DEM): slope and aspect. Both are converted to a percent-suitable-area measure per parcel, the same way land cover was above.

### Slope

Slopes at or below 10 degrees are treated as suitable. A threshold in this range is commonly used in utility-scale solar siting as a practical constructability limit, though the exact cutoff varies by developer and by whether the array uses fixed-tilt or terrain-following mounting.

In [6]:
DEM = Raster(r"C:\path\to\your\digital_elevation_model_30m")

slope = Slope(DEM, output_measurement="DEGREE", method="PLANAR")
slope.save(f"{output_gdb}\\slope")

In [7]:
#calculate statistics
arcpy.management.CalculateStatistics(slope)

<Result ''>

In [ ]:
slope_bins = Reclassify(
    f"{output_gdb}\\slope",
    "Value",
    RemapRange([
        [0, 10, 1],   # suitable
        [10, 90, 0]   # unsuitable
    ])
)

slope_bins.save(f"{output_gdb}\\slope_suitable")

In [43]:
slope_tabulate = f"{output_gdb}\\parcel_slope_tabulate"

TabulateArea(
    in_zone_data=parcels,
    zone_field="OBJECTID",
    in_class_data=f"{output_gdb}\\slope_suitable",
    class_field="Value",
    out_table=slope_tabulate,
    processing_cell_size=30)

<Result ''>

In [45]:
arcpy.management.AddField(
    slope_tabulate,
    "pct_slope_ok",
    "DOUBLE")

<Result 'parcel_slope_tabulate'>

In [47]:
expression = "(!VALUE_1! / (!VALUE_0! + !VALUE_1!)) * 100"

arcpy.management.CalculateField(
    slope_tabulate,
    "pct_slope_ok",
    expression,
    "PYTHON3"
)

<Result 'parcel_slope_tabulate'>

In [48]:
arcpy.management.JoinField(
    in_data=parcels,
    in_field="OBJECTID",
    join_table=slope_tabulate,
    join_field="OBJECTID",
    fields=["pct_slope_ok"]
)

<Result 'parcels_30acres'>

### Aspect

South-facing terrain (135°-225°) is treated as suitable, since it receives the most consistent annual solar exposure in the Northern Hemisphere.

**Bug fix:** the original run of this step hit a `RuntimeError` - the output raster already existed and overwrite was disabled, even though `overwriteOutput` had been set earlier in the session. That's a known ArcGIS Pro notebook quirk: environment settings don't always persist reliably across a long session, especially after a kernel restart. Setting it again immediately before the save, as below, avoids depending on that persistence.

In [8]:
aspect = Aspect(DEM)
aspect.save(f"{output_gdb}\\aspect")

In [ ]:
arcpy.env.overwriteOutput = True  # set again defensively, see note above

aspect = Raster(f"{output_gdb}\\aspect")

aspect_south_bin = Con(
    (aspect >= 135) & (aspect <= 225),
    1,
    0
)

aspect_south_bin.save(f"{output_gdb}\\aspect_south_bin")

In [56]:
aspect_tab = f"{output_gdb}\\parcel_aspect_tab_final"

TabulateArea(
    in_zone_data=parcels,
    zone_field="OBJECTID",
    in_class_data=f"{output_gdb}\\aspect_south_bin",
    class_field="Value",
    out_table=aspect_tab,
    processing_cell_size=30)

<Result ''>

In [57]:
arcpy.management.AddField(
    aspect_tab,
    "pct_south",
    "DOUBLE"
)

expression = "(!VALUE_1! / (!VALUE_0! + !VALUE_1!)) * 100"

arcpy.management.CalculateField(
    aspect_tab,
    "pct_south",
    expression,
    "PYTHON3")

<Result 'parcel_aspect_tab_final'>

In [58]:
arcpy.management.JoinField(
    in_data=parcels,
    in_field="OBJECTID",
    join_table=aspect_tab,
    join_field="OBJECTID",
    fields=["pct_south"]
)

<Result 'parcels_30acres'>

## Part 4: Road-density penalty

Parcels with a lot of major road running through them relative to their size are penalized, since heavily bisected parcels are more fragmented and harder to develop as a single array. Road length inside each parcel is intersected and summed, then converted to a relative penalty score: road kilometers per parcel square-kilometer, scaled by a factor of 25 and capped at 100. That scaling factor is a modeling choice, not a standardized unit - it was chosen so the penalty lands in a comparable range to the 0-100 percent-suitability scores above, not derived from any external source.

In [62]:
roads = r"C:\path\to\your\tl_2023_27_prisecroads.shp"

inter_fc = f"{output_gdb}\\parcel_roads_intersect"
arcpy.analysis.Intersect([parcels, roads], inter_fc)

arcpy.management.AddField(inter_fc, "road_m", "DOUBLE")
arcpy.management.CalculateGeometryAttributes(
    inter_fc,
    [["road_m", "LENGTH"]],
    length_unit="METERS"
)

sum_tbl = f"{output_gdb}\\parcel_roadlen_tbl"
arcpy.analysis.Statistics(
    inter_fc,
    sum_tbl,
    [["road_m", "SUM"]],
    case_field="OBJECTID"  # assumes OBJECTID carried through the intersect; check your own field name if not
)

arcpy.management.JoinField(parcels, "OBJECTID", sum_tbl, "OBJECTID", ["SUM_road_m"])

<Result 'parcels_30acres'>

In [63]:
arcpy.management.AddField(parcels, "road_inside_penalty", "DOUBLE")

# road km per parcel km², scaled by 25 and capped at 100 (see note above on this scaling choice)
expression = """
min(
    100,
    (
        (!SUM_road_m! / 1000) /
        (!acres_poly! * 0.00404686)
    ) * 25
) if !SUM_road_m! and !acres_poly! else 0
"""

arcpy.management.CalculateField(
    parcels,
    "road_inside_penalty",
    expression,
    "PYTHON3")

<Result 'parcels_30acres'>

## Part 5: Weighted suitability score

The three suitability percentages and the road penalty are combined into one score via a weighted linear combination:

```
final_score = (pct_suitable × 0.45) + (pct_slope_ok × 0.30) + (pct_south × 0.10) - (road_inside_penalty × 0.15)
```

**On the weights themselves, stated plainly:** these values (45/30/10/15) reflect a subjective prioritization - land cover suitability weighted as the dominant factor, slope next, aspect and road penalty treated as smaller adjustments - rather than a value derived from literature or a formal weight-elicitation method like pairwise Analytic Hierarchy Process (AHP) comparison. That's a real limitation of the model as it stands. The sensitivity analysis in Part 7 tests how much the actual top-10 result depends on that choice, rather than just presenting the ranking as if the weights were exact.

In [67]:
arcpy.management.AddField(parcels, "final_score", "DOUBLE")

expression = """
(
    (( !pct_suitable! or 0 ) * 0.45) +
    (( !pct_slope_ok! or 0 ) * 0.30) +
    (( !pct_south! or 0 ) * 0.10)
) - (
    (( !road_inside_penalty! or 0 ) * 0.15)
)
"""

arcpy.management.CalculateField(
    parcels,
    "final_score",
    expression,
    "PYTHON3"
)

<Result 'parcels_30acres'>

## Part 6: Ranking and selecting the top candidate sites

In [70]:
sorted_fc = f"{output_gdb}\\final_parcels_ranked"

arcpy.management.Sort(
    parcels,
    sorted_fc,
    [["final_score", "DESCENDING"]]
)

top10 = f"{output_gdb}\\top10_solar_sites"
top10_oids = []

with arcpy.da.SearchCursor(
    sorted_fc,
    ["OBJECTID"],
    sql_clause=(None, "ORDER BY final_score DESC")
) as cursor:
    for i, row in enumerate(cursor):
        if i == 10:
            break
        top10_oids.append(row[0])

oid_list = ",".join(map(str, top10_oids))
where = f"OBJECTID IN ({oid_list})"

arcpy.management.MakeFeatureLayer(sorted_fc, "final_rank_lyr")
arcpy.management.SelectLayerByAttribute("final_rank_lyr", "NEW_SELECTION", where)
arcpy.management.CopyFeatures("final_rank_lyr", top10)

<Result ''>

## Part 7: Sensitivity analysis - how stable is the top-10 list?

A weighted score is only as trustworthy as its weights are stable. Since the weights above were chosen subjectively rather than derived from literature or AHP, the honest next step is to check how much the actual result - the top-10 ranking - depends on that specific choice of weights, rather than presenting one ranking as if it were the only reasonable answer.

The approach: export the scored parcel table, then recompute `final_score` under several alternative, still-reasonable weight scenarios (the baseline plus perturbations of roughly ±20%, renormalized so the three positive weights still sum to 1). For each scenario, this reports what fraction of the original top 10 survives, and the Spearman rank correlation between the baseline ranking and the alternative ranking across *all* parcels - a low correlation or low overlap would mean the results are too sensitive to the specific weights chosen to be trusted; a high one means the ranking is robust to reasonable disagreement about exactly how much each factor should count.

In [ ]:
# This section requires exporting parcels_30acres (with pct_suitable, pct_slope_ok, pct_south,
# and road_inside_penalty already joined) to a table pandas can read - e.g. via
# arcpy.conversion.TableToTable(parcels, output_gdb, "parcels_for_sensitivity") and then
# arcpy.da.TableToNumPyArray(...) into the DataFrame below. Not yet run against real output -
# verify column names match your actual attribute table before trusting these numbers.

import pandas as pd
import numpy as np
from scipy.stats import spearmanr

# df = pd.DataFrame(arcpy.da.TableToNumPyArray(
#     f"{output_gdb}\\parcels_for_sensitivity",
#     ["OBJECTID", "pct_suitable", "pct_slope_ok", "pct_south", "road_inside_penalty"],
#     skip_nulls=True
# ))

baseline_weights = {"pct_suitable": 0.45, "pct_slope_ok": 0.30, "pct_south": 0.10, "penalty": 0.15}

# Perturbation scenarios: shift each positive weight up/down ~20%, renormalize the three
# positive weights to sum to 1, and independently perturb the penalty weight.
scenarios = {
    "baseline":              {"pct_suitable": 0.45, "pct_slope_ok": 0.30, "pct_south": 0.10, "penalty": 0.15},
    "landcover_up_20pct":    {"pct_suitable": 0.54, "pct_slope_ok": 0.27, "pct_south": 0.09, "penalty": 0.15},
    "landcover_down_20pct":  {"pct_suitable": 0.36, "pct_slope_ok": 0.33, "pct_south": 0.11, "penalty": 0.15},
    "slope_up_20pct":        {"pct_suitable": 0.41, "pct_slope_ok": 0.36, "pct_south": 0.09, "penalty": 0.15},
    "penalty_up_50pct":      {"pct_suitable": 0.45, "pct_slope_ok": 0.30, "pct_south": 0.10, "penalty": 0.225},
}

def score(df, w):
    return (
        df["pct_suitable"].fillna(0) * w["pct_suitable"]
        + df["pct_slope_ok"].fillna(0) * w["pct_slope_ok"]
        + df["pct_south"].fillna(0) * w["pct_south"]
        - df["road_inside_penalty"].fillna(0) * w["penalty"]
    )

# baseline_rank = score(df, scenarios["baseline"]).rank(ascending=False)
# baseline_top10 = set(df.loc[baseline_rank <= 10, "OBJECTID"])
#
# results = []
# for name, w in scenarios.items():
#     alt_rank = score(df, w).rank(ascending=False)
#     alt_top10 = set(df.loc[alt_rank <= 10, "OBJECTID"])
#     overlap = len(baseline_top10 & alt_top10) / 10
#     corr, _ = spearmanr(baseline_rank, alt_rank)
#     results.append({"scenario": name, "top10_overlap": overlap, "rank_correlation": corr})
#
# pd.DataFrame(results)

## Limitations & next steps

Stated together, rather than scattered as individual asides:

- **County coverage**: only 59 of 87 Minnesota counties are represented in the parcel dataset used here. Results are a top 10 among participating counties, not a true statewide top 10.
- **Subjective weighting**: the 45/30/10/15 weighting reflects a reasonable but subjective prioritization, not a literature-derived or AHP-elicited set of weights. Part 7 above is the honest check on how much that matters - run it against the real output before trusting the ranking too far.
- **Unverified reproducibility on the land cover step**: the code version of the NLCD reclassification in Part 1 hasn't been independently confirmed to produce output identical to the original manual pass.
- **No grid/interconnection factor**: proximity to substations and transmission capacity is often the single biggest real-world constraint on utility-scale solar siting, and isn't represented in this model at all. Adding it would likely reshuffle the rankings more than any of the weighting choices above.
- **No ground-truthing**: the top-10 sites haven't been checked against aerial imagery, ownership records, or any on-the-ground constraint (wetlands, cultural resources, existing easements) beyond what's captured in the input layers.